In [1]:
"""
Step 1 of 2: build the time series and save as CSV. No statistics here --
just turning the raw BAT and NVD files into clean daily/weekly counts.

Run this once. Then run analyze_q1_timeseries.py (step 2) against the CSVs
it produces, as many times as you want, with whatever lag range or stats
you want to try -- without re-reading the big raw files each time.

Outputs:
  q1_daily_series.csv    one row per calendar day, 2018-01-01 to present
  q1_weekly_series.csv   same, resampled to weekly sums

Columns in both:
  n_bat_ge1   posts that day/week with bat_score >= 1
  n_bat_ge2   posts that day/week with bat_score >= 2
  n_cve_run1  CVEs that day/week with cvss_base_score > 9.5
  n_cve_run2  CVEs that day/week with cvss_base_score >= 10
"""

import pandas as pd
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
BAT_PATH = "/Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates.csv"
NVD_PATH = "/Users/nadia/Desktop/redditRun_june/nvd_chunks/nvd_merged_full.csv"
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q1_analysis/"

BAT_THRESHOLDS = [1, 2]


def build_daily_bat(bat_path, thresholds):
    df = pd.read_csv(bat_path, usecols=["post_id", "bat_score", "created_date"])
    df["date"] = pd.to_datetime(df["created_date"], errors="coerce").dt.date

    print("bat_score distribution (sanity check):")
    print(df["bat_score"].value_counts().sort_index().to_string())

    agg = {}
    for t in thresholds:
        col = f"n_bat_ge{t}"
        df[col] = (df["bat_score"] >= t).astype(int)
        agg[col] = (col, "sum")

    daily = df.groupby("date").agg(**agg).reset_index()
    return daily


def build_daily_nvd(nvd_path):
    df = pd.read_csv(nvd_path, usecols=["cve_id", "published", "cvss_base_score"])
    df["date"] = pd.to_datetime(df["published"], errors="coerce").dt.date

    df["n_cve_run1"] = (df["cvss_base_score"] > 9.5).astype(int)
    df["n_cve_run2"] = (df["cvss_base_score"] >= 10).astype(int)

    daily = df.groupby("date").agg(
        n_cve_run1=("n_cve_run1", "sum"),
        n_cve_run2=("n_cve_run2", "sum"),
    ).reset_index()
    return daily


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    bat_cols = [f"n_bat_ge{t}" for t in BAT_THRESHOLDS]
    cve_cols = ["n_cve_run1", "n_cve_run2"]

    print("Building daily BAT series...")
    daily_bat = build_daily_bat(BAT_PATH, BAT_THRESHOLDS)

    print("\nBuilding daily NVD series...")
    daily_nvd = build_daily_nvd(NVD_PATH)

    # Combine on a continuous daily calendar, zero-filling quiet days
    start = min(daily_bat["date"].min(), daily_nvd["date"].min())
    end = max(daily_bat["date"].max(), daily_nvd["date"].max())
    full_index = pd.DataFrame({"date": pd.date_range(start, end, freq="D").date})

    daily = full_index.merge(daily_bat, on="date", how="left") \
                       .merge(daily_nvd, on="date", how="left")

    for c in bat_cols + cve_cols:
        daily[c] = daily[c].fillna(0)

    print(f"\nFull daily series: {len(daily)} days, {start} to {end}")

    daily_path = os.path.join(OUT_DIR, "q1_daily_series.csv")
    daily.to_csv(daily_path, index=False)
    print(f"Saved -> {daily_path}")

    # Weekly resample
    weekly = daily.copy()
    weekly["date"] = pd.to_datetime(weekly["date"])
    weekly = weekly.set_index("date")[bat_cols + cve_cols].resample("W").sum().reset_index()

    weekly_path = os.path.join(OUT_DIR, "q1_weekly_series.csv")
    weekly.to_csv(weekly_path, index=False)
    print(f"Saved -> {weekly_path}")

    print("\nDone. Run analyze_q1_timeseries.py next to do the correlation + lag scan.")


if __name__ == "__main__":
    main()

Building daily BAT series...
bat_score distribution (sanity check):
bat_score
0    108090
1     17398
2      6400
3      3373
4       636

Building daily NVD series...

Full daily series: 3071 days, 2018-01-01 to 2026-05-29
Saved -> /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_daily_series.csv
Saved -> /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_weekly_series.csv

Done. Run analyze_q1_timeseries.py next to do the correlation + lag scan.
